In [ ]:
import os
from datasets import load_dataset
from huggingface_hub import login
import json

HF_TOKEN = ""
REPO_ID = "mandal437/bert-token-detector-bio"

os.environ["HF_TOKEN"] = HF_TOKEN.strip()
login(token=HF_TOKEN.strip(), add_to_git_credential=False)

ds = load_dataset(REPO_ID, token=True)
print(ds)
print(len(ds["train"]), len(ds["validation"]), len(ds["test"]))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 225608
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 26330
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 28317
    })
})
225608 26330 28317


In [2]:
from pathlib import Path
WORKDIR = Path.cwd()

In [3]:
from train_token_classifier import run_training, TrainConfig

STAGE1_OUT = WORKDIR / "token_detector_stage1_head_only"

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-04-27 07:10:57.303656: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [ ]:
from train_token_classifier import run_training, TrainConfig

STAGE1_OUT = WORKDIR / "token_detector_stage1_head_only"
config = TrainConfig(
    hf_dataset_dir="",                 # не используем путь
    bio_data_dir="",                   # не используем bio jsonl
    model_name="microsoft/mdeberta-v3-base",
    output_dir=str(STAGE1_OUT),

    train_mode="head_only",
    partial_unfreeze_last_n_layers=2,

    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    max_grad_norm=1.0,

    logging_steps=50,
    save_total_limit=1,
    dataloader_num_workers=2,
    seed=42,

    fp16=False,
    bf16=False,
    cpu_only=False,
    gradient_checkpointing=False,
    group_by_length=True,

    use_focal_loss=True,
    focal_gamma=1.5,

    use_class_balanced_alpha=True,
    effective_num_beta=0.9999,
    alpha_clip_max=10.0,
    manual_alpha=None,

    use_weighted_sampler=True,
    sampler_human_weight=1.0,
    sampler_ai_weight=1.5,
    sampler_bai_weight=4.0,

    eval_strategy="epoch",
    save_strategy="no",
    eval_steps=500,
    save_steps=500,
    load_best_model_at_end=False,
)

run_training(config, dataset_override=ds)


Unknown instance spec: Please select VM configuration

In [4]:
import json

for name in [
    "run_metadata.json",
    "train_metrics.json",
    "val_metrics.json",
    "test_metrics.json",
    "training_summary.json",
]:
    path = STAGE1_OUT / name
    print(f"\n===== {name} =====")
    with open(path, "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), ensure_ascii=False, indent=2)[:4000])



===== run_metadata.json =====
{
  "run_config": {
    "hf_dataset_dir": "",
    "bio_data_dir": "",
    "model_name": "microsoft/mdeberta-v3-base",
    "output_dir": "/home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_stage1_head_only",
    "train_mode": "head_only",
    "partial_unfreeze_last_n_layers": 2,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 0.0003,
    "weight_decay": 0.01,
    "warmup_ratio": 0.03,
    "warmup_steps": 0,
    "lr_scheduler_type": "linear",
    "max_grad_norm": 1.0,
    "logging_steps": 50,
    "save_total_limit": 1,
    "dataloader_num_workers": 2,
    "seed": 42,
    "fp16": false,
    "bf16": false,
    "cpu_only": false,
    "gradient_checkpointing": false,
    "group_by_length": true,
    "use_focal_loss": true,
    "focal_gamma": 1.5,
    "use_class_balanced_alpha": true,
    "effective_num_beta": 0.9999,
    "alp

In [5]:
for name in [
    "val_classification_report.json",
    "test_classification_report.json",
]:
    path = STAGE1_OUT / name
    print(f"\n===== {name} =====")
    with open(path, "r", encoding="utf-8") as f:
        report = json.load(f)

    short = {
        k: report[k]
        for k in ["O", "B-AI", "I-AI", "accuracy", "macro avg", "weighted avg"]
        if k in report
    }
    print(json.dumps(short, ensure_ascii=False, indent=2))



===== val_classification_report.json =====
{
  "O": {
    "precision": 0.8686066864532294,
    "recall": 0.14803438259112947,
    "f1-score": 0.2529578205305026,
    "support": 5011606.0
  },
  "B-AI": {
    "precision": 0.4733851108164425,
    "recall": 0.6851267991775188,
    "f1-score": 0.5599058981683751,
    "support": 14590.0
  },
  "I-AI": {
    "precision": 0.4966760501908923,
    "recall": 0.9742026727048351,
    "f1-score": 0.6579239036266211,
    "support": 4317424.0
  },
  "accuracy": 0.5306221785560629,
  "macro avg": {
    "precision": 0.6128892824868547,
    "recall": 0.6024546181578277,
    "f1-score": 0.49026254077516623,
    "support": 9343620.0
  },
  "weighted avg": {
    "precision": 0.6961308646493862,
    "recall": 0.5306221785560629,
    "f1-score": 0.44056055467400507,
    "support": 9343620.0
  }
}

===== test_classification_report.json =====
{
  "O": {
    "precision": 0.8480652369660426,
    "recall": 0.1351442517448422,
    "f1-score": 0.2331367693183234,


In [6]:
STAGE2_OUT = WORKDIR / "token_detector_stage2_partial_unfreeze"
STAGE1_MODEL = STAGE1_OUT / "final_model"

In [7]:
config = TrainConfig(
    hf_dataset_dir="",
    bio_data_dir="",
    model_name=str(STAGE1_MODEL),
    output_dir=str(STAGE2_OUT),

    train_mode="partial_unfreeze",
    partial_unfreeze_last_n_layers=2,

    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-6,
    weight_decay=0.01,
    warmup_ratio=0.02,
    max_grad_norm=0.5,

    logging_steps=50,
    save_total_limit=1,
    dataloader_num_workers=2,
    seed=42,

    fp16=False,
    bf16=False,
    cpu_only=False,
    gradient_checkpointing=False,
    group_by_length=True,

    use_focal_loss=True,
    focal_gamma=1.5,

    use_class_balanced_alpha=False,
    effective_num_beta=0.9999,
    alpha_clip_max=10.0,
    manual_alpha=None,

    use_weighted_sampler=False,
    sampler_human_weight=1.0,
    sampler_ai_weight=1.5,
    sampler_bai_weight=4.0,

    eval_strategy="epoch",
    save_strategy="no",
    eval_steps=500,
    save_steps=500,
    load_best_model_at_end=False,
)

run_training(config, dataset_override=ds)


 15%|█▍        | 4150/28201 [15:06<1:27:03,  4.60it/s]

{'loss': 0.1263, 'grad_norm': 5.191019535064697, 'learning_rate': 2.6108336951801995e-06, 'epoch': 0.15}


 15%|█▍        | 4200/28201 [15:17<1:27:07,  4.59it/s]

{'loss': 0.1117, 'grad_norm': 7.83155632019043, 'learning_rate': 2.605405992184108e-06, 'epoch': 0.15}


 15%|█▌        | 4250/28201 [15:28<1:24:56,  4.70it/s]

...

[TRAIN] Evaluating on test...



100%|██████████| 14159/14159 [10:04<00:00, 23.41it/s]


[TRAIN] Building classification reports...


100%|██████████| 14159/14159 [10:02<00:00, 23.50it/s]


[TRAIN] Saving final model + tokenizer...

=== TRAINING FINISHED ===
Final model: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_stage2_partial_unfreeze/final_model
Training summary: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_stage2_partial_unfreeze/training_summary.json

Validation metrics:
{
  "val_loss": 0.09216444194316864,
  "val_token_precision_macro": 0.8853449097768613,
  "val_token_recall_macro": 0.8793861042337676,
  "val_token_f1_macro": 0.8808574059316282,
  "val_token_precision_weighted": 0.9280798712419057,
  "val_token_recall_weighted": 0.9235613177761938,
  "val_token_f1_weighted": 0.9236478343412942,
  "val_ai_token_precision": 0.8782558115474546,
  "val_ai_token_recall": 0.9699876316189191,
  "val_ai_token_f1": 0.921845309662253,
  "val_seqeval_precision": 0.08586951871657754,
  "val_seqeval_recall": 0.6878684030157642,
  "val_seqeval_f1": 0.15267942037804738,
  "val_class_O_precision": 0.9714827608821581,


In [8]:
for name in [
    "run_metadata.json",
    "val_metrics.json",
    "test_metrics.json",
    "training_summary.json",
]:
    path = STAGE2_OUT / name
    print(f"\n===== {name} =====")
    with open(path, "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), ensure_ascii=False, indent=2)[:4000])



===== run_metadata.json =====
{
  "run_config": {
    "hf_dataset_dir": "",
    "bio_data_dir": "",
    "model_name": "/home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_stage1_head_only/final_model",
    "output_dir": "/home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_stage2_partial_unfreeze",
    "train_mode": "partial_unfreeze",
    "partial_unfreeze_last_n_layers": 2,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 3e-06,
    "weight_decay": 0.01,
    "warmup_ratio": 0.02,
    "warmup_steps": 0,
    "lr_scheduler_type": "linear",
    "max_grad_norm": 0.5,
    "logging_steps": 50,
    "save_total_limit": 1,
    "dataloader_num_workers": 2,
    "seed": 42,
    "fp16": false,
    "bf16": false,
    "cpu_only": false,
    "gradient_checkpointing": false,
    "group_by_length": true,
    "use_focal_loss": true,
    "focal_ga